In [1]:
import os
import torch
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm, trange
import argparse
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from dataset_swat import ForecastSimpleDataloader
from code_test import test

import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from fddbenchmark import FDDDataset
from dataset3 import ForecastFDDDataloader
import numpy as np
from tqdm.auto import tqdm, trange

In [3]:
model = TimeSeriesTransformerGSL(
        ts_dim=51,
        window_size=100,
        d_model=256,
        nhead=8,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.2,
        gsl_k=7,
        n_gnn=1,
        n_hidden=256,
        device='cpu',
    )

In [4]:
m = torch.load('/home/akozhevnikov/graphs/ts2/saved_models/swat_transformer_gsl_base.pt', weights_only=True)
model.load_state_dict(m)

<All keys matched successfully>

In [5]:
df_att  = pd.read_csv('SWaT_Dataset_Attack_v0.csv')
feature_cols = [c for c in df_att.columns if c not in ['Timestamp', 'Normal/Attack']]

df_norm = pd.read_csv('SWaT_Dataset_Normal_v0.csv')
scaler = StandardScaler()
scaler.fit(df_norm[feature_cols])

df_test = df_att
df_test[feature_cols] = df_att[feature_cols].astype('float32')
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

df_test['Normal/Attack'] = df_test['Normal/Attack'].apply(lambda x: 0 if x=='Normal' else 1)

In [6]:
test_dl = ForecastSimpleDataloader(
    dataframe=df_test,
    window_size=100,
    step_size=1,
    use_minibatches=True,
    batch_size=512,
    shuffle=False,
    data_framework='torch',
    device='cpu',
    train=False,             # используем все окна, в том числе с атаками
    timestamp_col='Timestamp',
    label_col='Normal/Attack',
    disable_index=False
)

In [7]:
all_errors = []
all_true_labels = []
all_errors_vec = []

for i in range(len(test_dl) - 1):
    ts_batch, _, target_batch, label = test_dl[i]
    pred = model(ts_batch)

    target_raw = scaler.inverse_transform(target_batch.cpu().detach().numpy())
    pred_raw = scaler.inverse_transform(pred.cpu().detach().numpy())

    errors = target_raw - pred_raw
    all_errors_vec.extend(errors)
    all_errors.extend(abs(errors).mean(axis=1))
    all_true_labels.extend(label)

In [18]:
ans

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report, f1_score

all_errors = np.stack(all_errors_vec)
all_true_labels = np.array(all_true_labels)

X_train, X_test, y_train, y_test = train_test_split(
    all_errors, all_true_labels,
    test_size=1-0.00667,
    shuffle=True,
    random_state=42,
)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    all_errors, all_true_labels,
    test_size=1-0.00667,
    shuffle=True, 
    random_state=i,
)
from catboost import CatBoostClassifier

c = CatBoostClassifier(iterations=10000, early_stopping_rounds=5, class_weights=(0.7, 0.3))
c.fit(X_train, y_train)
pred = c.predict(X_test)

Learning rate set to 0.001993
0:	learn: 0.6896994	total: 1.63ms	remaining: 16.3s
1:	learn: 0.6861719	total: 2.72ms	remaining: 13.6s
2:	learn: 0.6826705	total: 3.76ms	remaining: 12.5s
3:	learn: 0.6792360	total: 4.81ms	remaining: 12s
4:	learn: 0.6758664	total: 5.81ms	remaining: 11.6s
5:	learn: 0.6725445	total: 6.84ms	remaining: 11.4s
6:	learn: 0.6692078	total: 7.86ms	remaining: 11.2s
7:	learn: 0.6658520	total: 8.9ms	remaining: 11.1s
8:	learn: 0.6624988	total: 9.94ms	remaining: 11s
9:	learn: 0.6591409	total: 11ms	remaining: 11s
10:	learn: 0.6559096	total: 14ms	remaining: 12.8s
11:	learn: 0.6525989	total: 17.2ms	remaining: 14.3s
12:	learn: 0.6492335	total: 19.2ms	remaining: 14.8s
13:	learn: 0.6459676	total: 20.3ms	remaining: 14.5s
14:	learn: 0.6427172	total: 21.4ms	remaining: 14.2s
15:	learn: 0.6394346	total: 22.4ms	remaining: 14s
16:	learn: 0.6363225	total: 23.5ms	remaining: 13.8s
17:	learn: 0.6332072	total: 26.2ms	remaining: 14.5s
18:	learn: 0.6300561	total: 28.1ms	remaining: 14.8s
19:	l

In [21]:
print(classification_report(y_test, pred))
print(len(X_train))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98    392285
           1       0.99      0.76      0.86     54253

    accuracy                           0.97    446538
   macro avg       0.98      0.88      0.92    446538
weighted avg       0.97      0.97      0.97    446538

2998


In [22]:
from catboost import Pool

In [23]:
model.gsl[0]
model.eval()


gnn_features =[]

with torch.no_grad():
    for i in range(len(test_dl) - 1):
        ts_batch, _, target_batch, label = test_dl[i]
        x_gnn = ts_batch.transpose(1, 2)
        with torch.no_grad():
            adj = model.gsl[0](torch.arange(51)) * model.z
            h = model.conv1[0](adj, x_gnn).relu()
            h = model.bnorm1[0](h)
            skip, _ = torch.min(h, dim=1)
            h = model.conv2[0](adj, h).relu()
            h = model.bnorm2[0](h)
            h, _ = torch.min(h, dim=1)
            h += skip
            gnn_features.extend(h)

In [24]:
gnn_features = np.stack(gnn_features)

In [25]:
all_errors = np.stack(all_errors_vec)

In [26]:
features = np.concatenate([gnn_features, all_errors], axis=1)

In [27]:
def augment_with_noise(X, noise_level=0.05, n_copies=1, random_state=None):
    rng = np.random.default_rng(random_state)
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    ranges = X_max - X_min

    augmented = []
    for _ in range(n_copies):
        noise = rng.uniform(
            low=-noise_level * ranges,
            high=+noise_level * ranges,
            size=X.shape
        )
        augmented.append(X + noise)
    return np.vstack(augmented)

def augment_with_masking(X, mask_prob=0.1, n_copies=1, random_state=None):
    rng = np.random.default_rng(random_state)
    augmented = []
    for _ in range(n_copies):
        mask = rng.random(size=X.shape) < mask_prob
        X_masked = X.copy()
        X_masked[mask] = 0.0
        augmented.append(X_masked)
    return np.vstack(augmented)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(   
    features, all_true_labels,
    test_size=1-0.00667,
    shuffle=True, 
    random_state=42,
)

In [29]:
noise_copies = 4
mask_copies = 4

X_noise = augment_with_noise(X_train, noise_level=0.05, n_copies=noise_copies, random_state=42)
X_mask  = augment_with_masking(X_train, mask_prob=0.1, n_copies=mask_copies, random_state=42)

X_train_aug = np.vstack([X_train, X_noise, X_mask])
y_train_aug = np.concatenate([
    y_train,
    np.tile(y_train, noise_copies),
    np.tile(y_train, mask_copies),
])

In [30]:
from catboost import CatBoostClassifier
c = CatBoostClassifier(iterations=10000,
                       auto_class_weights='Balanced', # balanced are best
                      )

c.fit(X_train, y_train)
pred = c.predict(X_test)

Learning rate set to 0.001993
0:	learn: 0.6907137	total: 10.3ms	remaining: 1m 43s
1:	learn: 0.6881286	total: 17.6ms	remaining: 1m 28s
2:	learn: 0.6855390	total: 26.7ms	remaining: 1m 28s
3:	learn: 0.6828240	total: 34.5ms	remaining: 1m 26s
4:	learn: 0.6808327	total: 41.4ms	remaining: 1m 22s
5:	learn: 0.6782789	total: 48.1ms	remaining: 1m 20s
6:	learn: 0.6758876	total: 55.5ms	remaining: 1m 19s
7:	learn: 0.6731909	total: 62.2ms	remaining: 1m 17s
8:	learn: 0.6707451	total: 68ms	remaining: 1m 15s
9:	learn: 0.6685398	total: 73.6ms	remaining: 1m 13s
10:	learn: 0.6663318	total: 79ms	remaining: 1m 11s
11:	learn: 0.6637518	total: 85.4ms	remaining: 1m 11s
12:	learn: 0.6612849	total: 93.9ms	remaining: 1m 12s
13:	learn: 0.6591487	total: 99.9ms	remaining: 1m 11s
14:	learn: 0.6574058	total: 105ms	remaining: 1m 9s
15:	learn: 0.6550510	total: 110ms	remaining: 1m 8s
16:	learn: 0.6529799	total: 120ms	remaining: 1m 10s
17:	learn: 0.6508783	total: 127ms	remaining: 1m 10s
18:	learn: 0.6486876	total: 137ms	re

In [31]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99    392262
           1       0.99      0.87      0.92     54276

    accuracy                           0.98    446538
   macro avg       0.98      0.93      0.95    446538
weighted avg       0.98      0.98      0.98    446538



In [32]:
len(X_train)

2998